# 🎬 TikTok A/B Experiment Analysis
### Simulating a real-world experimentation suite across 4 concurrent product tests

**Author:** Kalpana Joyce Dovari  
**Dataset:** Synthetic — 50,000 users, distributions calibrated to publicly reported TikTok benchmarks  
**Methods:** Welch's T-Test · Chi-Square Test · Cohen's d · Confidence Intervals  

---

## Experiment Overview

| # | Experiment | Metric | Test Used | Expected Result |
|---|---|---|---|---|
| 1 | New Recommendation Algorithm | Watch Time (seconds) | Welch's T-Test | ✅ Significant |
| 2 | New Thumbnail Format | Click-Through Rate | Chi-Square Test | ✅ Significant |
| 3 | New Notification Timing | App Opens | Welch's T-Test | ❌ Not Significant |
| 4 | Autoplay Toggle | Session Length (minutes) | Welch's T-Test | ❌ Not Significant |

> **Note on synthetic data:** Real TikTok/YouTube experiment data is proprietary. 
> This dataset was generated using NumPy distributions (Normal for continuous metrics, 
> Binomial for binary outcomes) with parameters grounded in industry benchmarks. 
> The methodology — experiment design, statistical testing, interpretation — is identical 
> to what's applied on real internal data.


## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
from scipy.stats import ttest_ind, chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999999',
    'ytick.color':      '#999999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
})

PINK   = '#fe2c55'   # TikTok red-pink
CYAN   = '#25f4ee'   # TikTok cyan
GOLD   = '#ffd700'
GREY   = '#888888'
GREEN  = '#00c853'
WHITE  = '#e0e0e0'

df = pd.read_csv('../data/tiktok_ab_experiment_data.csv')
print(f"Dataset shape: {df.shape}")
df.head()


## 2. Exploratory Data Analysis

In [ ]:
# User demographics
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('User Demographics', fontsize=14, color=WHITE, fontweight='bold')

# Age
axes[0].hist(df['age'], bins=30, color=PINK, alpha=0.85, edgecolor='#0f0f0f')
axes[0].set_title('Age Distribution', color=WHITE)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Country
country_counts = df['country'].value_counts()
axes[1].bar(country_counts.index, country_counts.values, color=CYAN, alpha=0.85)
axes[1].set_title('Country Distribution', color=WHITE)
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Users')

# Device
device_counts = df['device'].value_counts()
axes[2].pie(device_counts.values, labels=device_counts.index,
            colors=[PINK, CYAN], autopct='%1.1f%%',
            textprops={'color': WHITE})
axes[2].set_title('Device Split', color=WHITE)

plt.tight_layout()
plt.show()
print(f"\nTotal users: {len(df):,}")
print(f"Avg age: {df['age'].mean():.1f}")
print(f"Avg tenure: {df['tenure_days'].mean():.0f} days")


## 3. Statistical Testing Functions

In [ ]:
def cohens_d(group_a, group_b):
    """Effect size for continuous metrics."""
    pooled_std = np.sqrt((group_a.std()**2 + group_b.std()**2) / 2)
    return (group_b.mean() - group_a.mean()) / pooled_std

def interpret_d(d):
    d = abs(d)
    if d < 0.2:   return 'Negligible'
    elif d < 0.5: return 'Small'
    elif d < 0.8: return 'Medium'
    else:         return 'Large'

def confidence_interval(group, confidence=0.95):
    n = len(group)
    mean = group.mean()
    se = stats.sem(group)
    h = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean - h, mean + h

def print_result(name, p_value, threshold=0.05):
    sig = p_value < threshold
    icon = '✅ SIGNIFICANT' if sig else '❌ NOT SIGNIFICANT'
    decision = 'SHIP IT 🚀' if sig else 'DO NOT SHIP ⛔'
    print(f"  p-value:    {p_value:.6f}")
    print(f"  Result:     {icon} (α = {threshold})")
    print(f"  Decision:   {decision}")
    return sig


## 4. Experiment 1 — New Recommendation Algorithm
**Hypothesis:** The new recommendation algorithm increases average watch time per video.  
**Metric:** `watch_time_seconds` (continuous)  
**Test:** Welch's T-Test (does not assume equal variance between groups)


In [ ]:
control   = df[df['exp1_group'] == 'control']['watch_time_seconds']
treatment = df[df['exp1_group'] == 'treatment']['watch_time_seconds']

t_stat, p_value = ttest_ind(control, treatment, equal_var=False)
d = cohens_d(control, treatment)
ci_c = confidence_interval(control)
ci_t = confidence_interval(treatment)

print("=" * 50)
print("EXPERIMENT 1: Recommendation Algorithm")
print("=" * 50)
print(f"  Control   mean: {control.mean():.2f}s  (n={len(control):,})")
print(f"  Treatment mean: {treatment.mean():.2f}s  (n={len(treatment):,})")
print(f"  Lift:          +{treatment.mean() - control.mean():.2f}s")
print(f"  t-statistic:   {t_stat:.4f}")
print(f"  Cohen's d:     {d:.4f} ({interpret_d(d)} effect)")
print_result("Exp 1", p_value)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Exp 1: Recommendation Algorithm — Watch Time', color=WHITE, fontsize=13, fontweight='bold')

# Distribution
axes[0].hist(control,   bins=60, alpha=0.6, color=CYAN,  label='Control',   density=True)
axes[0].hist(treatment, bins=60, alpha=0.6, color=PINK,  label='Treatment', density=True)
axes[0].axvline(control.mean(),   color=CYAN,  linestyle='--', linewidth=2)
axes[0].axvline(treatment.mean(), color=PINK,  linestyle='--', linewidth=2)
axes[0].set_title('Watch Time Distribution', color=WHITE)
axes[0].set_xlabel('Watch Time (seconds)')
axes[0].set_ylabel('Density')
axes[0].legend(facecolor='#1a1a1a', labelcolor=WHITE)

# CI plot
groups = ['Control', 'Treatment']
means  = [control.mean(), treatment.mean()]
errors = [control.mean() - ci_c[0], treatment.mean() - ci_t[0]]
colors = [CYAN, PINK]
bars = axes[1].bar(groups, means, color=colors, alpha=0.8, width=0.4)
axes[1].errorbar(groups, means, yerr=errors, fmt='none', color=WHITE, capsize=8, linewidth=2)
axes[1].set_title('Mean Watch Time ± 95% CI', color=WHITE)
axes[1].set_ylabel('Watch Time (seconds)')
for bar, mean in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, mean + 1,
                 f'{mean:.1f}s', ha='center', color=WHITE, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Experiment 2 — New Thumbnail Format
**Hypothesis:** The new large thumbnail format increases click-through rate (CTR).  
**Metric:** `clicked` (binary: 0 or 1)  
**Test:** Chi-Square Test (used for categorical / binary outcomes)


In [ ]:
control_c   = df[df['exp2_group'] == 'control']['clicked']
treatment_c = df[df['exp2_group'] == 'treatment']['clicked']

# Contingency table
ct = pd.crosstab(df['exp2_group'], df['clicked'])
chi2, p_value, dof, expected = chi2_contingency(ct)

ctrl_ctr  = control_c.mean() * 100
treat_ctr = treatment_c.mean() * 100

print("=" * 50)
print("EXPERIMENT 2: Thumbnail Format (CTR)")
print("=" * 50)
print(f"  Control   CTR: {ctrl_ctr:.2f}%  (n={len(control_c):,})")
print(f"  Treatment CTR: {treat_ctr:.2f}%  (n={len(treatment_c):,})")
print(f"  Lift:          +{treat_ctr - ctrl_ctr:.2f}pp")
print(f"  Chi² stat:     {chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print_result("Exp 2", p_value)

# Contingency table display
print(f"\nContingency Table:")
print(ct.to_string())

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Exp 2: Thumbnail Format — Click-Through Rate', color=WHITE, fontsize=13, fontweight='bold')

# CTR bar
bars = axes[0].bar(['Control', 'Treatment'], [ctrl_ctr, treat_ctr],
                   color=[CYAN, PINK], alpha=0.85, width=0.4)
axes[0].set_title('CTR by Group', color=WHITE)
axes[0].set_ylabel('Click-Through Rate (%)')
for bar, val in zip(bars, [ctrl_ctr, treat_ctr]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f'{val:.2f}%', ha='center', color=WHITE, fontweight='bold')

# Stacked proportion
props = ct.div(ct.sum(axis=1), axis=0) * 100
props.plot(kind='bar', stacked=True, ax=axes[1],
           color=['#1a1a1a', PINK], edgecolor='#333', rot=0)
axes[1].set_title('Click Proportion by Group', color=WHITE)
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(['Not Clicked', 'Clicked'], facecolor='#1a1a1a', labelcolor=WHITE)

plt.tight_layout()
plt.show()


## 6. Experiment 3 — New Notification Timing
**Hypothesis:** Sending push notifications at a different time increases daily app opens.  
**Metric:** `app_opens` (continuous)  
**Test:** Welch's T-Test  
> ⚠️ This experiment was designed with **no real effect** — the result should NOT be significant.


In [ ]:
control_o   = df[df['exp3_group'] == 'control']['app_opens']
treatment_o = df[df['exp3_group'] == 'treatment']['app_opens']

t_stat, p_value = ttest_ind(control_o, treatment_o, equal_var=False)
d = cohens_d(control_o, treatment_o)

print("=" * 50)
print("EXPERIMENT 3: Notification Timing (App Opens)")
print("=" * 50)
print(f"  Control   mean: {control_o.mean():.3f}  (n={len(control_o):,})")
print(f"  Treatment mean: {treatment_o.mean():.3f}  (n={len(treatment_o):,})")
print(f"  Difference:    {treatment_o.mean() - control_o.mean():.4f}")
print(f"  t-statistic:   {t_stat:.4f}")
print(f"  Cohen's d:     {d:.4f} ({interpret_d(d)} effect)")
print_result("Exp 3", p_value)

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(control_o,   bins=50, alpha=0.6, color=CYAN, label='Control',   density=True)
ax.hist(treatment_o, bins=50, alpha=0.6, color=PINK, label='Treatment', density=True)
ax.axvline(control_o.mean(),   color=CYAN, linestyle='--', linewidth=2)
ax.axvline(treatment_o.mean(), color=PINK, linestyle='--', linewidth=2)
ax.set_title('Exp 3: App Opens Distribution — Groups Almost Identical', color=WHITE, fontsize=13)
ax.set_xlabel('Daily App Opens')
ax.set_ylabel('Density')
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE)
plt.tight_layout()
plt.show()


## 7. Experiment 4 — Autoplay Toggle
**Hypothesis:** Enabling autoplay by default increases session length.  
**Metric:** `session_length_minutes` (continuous)  
**Test:** Welch's T-Test  
> ⚠️ This experiment was also designed with **no real effect**.


In [ ]:
control_s   = df[df['exp4_group'] == 'control']['session_length_minutes']
treatment_s = df[df['exp4_group'] == 'treatment']['session_length_minutes']

t_stat, p_value = ttest_ind(control_s, treatment_s, equal_var=False)
d = cohens_d(control_s, treatment_s)

print("=" * 50)
print("EXPERIMENT 4: Autoplay Toggle (Session Length)")
print("=" * 50)
print(f"  Control   mean: {control_s.mean():.3f} min  (n={len(control_s):,})")
print(f"  Treatment mean: {treatment_s.mean():.3f} min  (n={len(treatment_s):,})")
print(f"  Difference:    {treatment_s.mean() - control_s.mean():.4f} min")
print(f"  t-statistic:   {t_stat:.4f}")
print(f"  Cohen's d:     {d:.4f} ({interpret_d(d)} effect)")
print_result("Exp 4", p_value)

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(control_s,   bins=50, alpha=0.6, color=CYAN, label='Control',   density=True)
ax.hist(treatment_s, bins=50, alpha=0.6, color=PINK, label='Treatment', density=True)
ax.axvline(control_s.mean(),   color=CYAN, linestyle='--', linewidth=2)
ax.axvline(treatment_s.mean(), color=PINK, linestyle='--', linewidth=2)
ax.set_title('Exp 4: Session Length Distribution — No Signal', color=WHITE, fontsize=13)
ax.set_xlabel('Session Length (minutes)')
ax.set_ylabel('Density')
ax.legend(facecolor='#1a1a1a', labelcolor=WHITE)
plt.tight_layout()
plt.show()


## 8. Executive Summary — All Experiments

In [ ]:
summary = {
    'Experiment': [
        'Exp 1: Recommendation Algorithm',
        'Exp 2: Thumbnail Format',
        'Exp 3: Notification Timing',
        'Exp 4: Autoplay Toggle',
    ],
    'Metric': ['Watch Time (s)', 'CTR (%)', 'App Opens', 'Session Length (min)'],
    'Test': ['Welch T-Test', 'Chi-Square', 'Welch T-Test', 'Welch T-Test'],
    'Control': [
        df[df['exp1_group']=='control']['watch_time_seconds'].mean(),
        df[df['exp2_group']=='control']['clicked'].mean() * 100,
        df[df['exp3_group']=='control']['app_opens'].mean(),
        df[df['exp4_group']=='control']['session_length_minutes'].mean(),
    ],
    'Treatment': [
        df[df['exp1_group']=='treatment']['watch_time_seconds'].mean(),
        df[df['exp2_group']=='treatment']['clicked'].mean() * 100,
        df[df['exp3_group']=='treatment']['app_opens'].mean(),
        df[df['exp4_group']=='treatment']['session_length_minutes'].mean(),
    ],
    'Decision': ['🚀 SHIP', '🚀 SHIP', '⛔ DO NOT SHIP', '⛔ DO NOT SHIP'],
}

summary_df = pd.DataFrame(summary)
summary_df['Lift'] = (
    (pd.Series(summary['Treatment']) - pd.Series(summary['Control']))
    / pd.Series(summary['Control']) * 100
).round(2).astype(str) + '%'
summary_df['Control'] = pd.Series(summary['Control']).round(3)
summary_df['Treatment'] = pd.Series(summary['Treatment']).round(3)

print("\n" + "="*70)
print("EXPERIMENT SUITE SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))
print("\nKey Insight: 2 of 4 experiments showed statistically significant")
print("effects. This mirrors realistic experimentation — not every product")
print("change improves the metric it targets.")


## 9. Key Takeaways

- **Experiment 1 (Recommendation Algorithm):** The new algorithm increased average watch time by ~12 seconds per video — a statistically and practically significant result. **Recommend shipping.**

- **Experiment 2 (Thumbnail Format):** CTR improved from ~4% to ~5.8%, a ~45% relative lift — highly significant. **Recommend shipping.**

- **Experiment 3 (Notification Timing):** No meaningful difference in app opens. The change in notification timing had no effect on re-engagement. **Do not ship.**

- **Experiment 4 (Autoplay Toggle):** Session length was statistically identical between groups. Autoplay default did not extend sessions. **Do not ship.**

### On statistical testing choices
- **Welch's T-Test** was chosen over Student's T-Test because we cannot assume equal variance between control and treatment groups in real user experiments.
- **Chi-Square** was used for the CTR experiment because the outcome is binary (clicked / not clicked) — not continuous.
- **Cohen's d** was computed for all continuous metrics to measure *practical* significance, not just statistical significance. A result can be statistically significant but too small to matter in practice.
- **α = 0.05** was used as the significance threshold throughout, meaning we accept a 5% chance of a false positive per test.
